In [2]:
!pip install datasets

In [3]:
from datasets import load_dataset

dataset = load_dataset("Ankita802/code-description1")

Extracting data files:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/395 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/99 [00:00<?, ? examples/s]

Dataset parquet downloaded and prepared to C:/Users/diya.sharma/.cache/huggingface/datasets/Ankita802___parquet/Ankita802--code-description1-29915d6434e4c048/0.0.0/2a3b91fbd88a2c90d1dbbb32b460cf621d31bd5b05b934492fdef7d8d6f236ec. Subsequent calls will reuse this data.


  0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
!pip install transformers

In [6]:
from transformers import FunnelForQuestionAnswering, FunnelTokenizer

In [7]:
tokenizer=FunnelTokenizer.from_pretrained("funnel-transformer/small-base")
model=FunnelForQuestionAnswering.from_pretrained("funnel-transformer/small-base")

Some weights of FunnelForQuestionAnswering were not initialized from the model checkpoint at funnel-transformer/small-base and are newly initialized: ['decoder.layers.0.attention.k_head.bias', 'qa_outputs.weight', 'decoder.layers.1.ffn.layer_norm.bias', 'decoder.layers.1.ffn.linear_1.bias', 'decoder.layers.0.ffn.linear_2.weight', 'decoder.layers.1.attention.r_kernel', 'decoder.layers.1.ffn.linear_2.bias', 'decoder.layers.0.ffn.linear_1.bias', 'decoder.layers.0.attention.k_head.weight', 'decoder.layers.1.attention.layer_norm.bias', 'decoder.layers.0.attention.r_s_bias', 'decoder.layers.0.ffn.linear_2.bias', 'decoder.layers.0.attention.r_w_bias', 'decoder.layers.0.attention.post_proj.bias', 'decoder.layers.1.attention.k_head.weight', 'decoder.layers.1.attention.post_proj.weight', 'decoder.layers.1.ffn.linear_1.weight', 'decoder.layers.0.attention.v_head.weight', 'decoder.layers.1.attention.r_r_bias', 'decoder.layers.0.ffn.linear_1.weight', 'decoder.layers.1.attention.r_w_bias', 'decoder.

In [66]:
# def tokenize_function(example):
#     # start_prompt = 'Describe the input query of user.\n\n'
#     # end_prompt = '\n\nDescription: '
# #     prompt = [dialogue for dialogue in example["input"]]
#     example['input_ids'] = tokenizer(example["code_snippet"], padding="max_length", truncation=True, return_tensors="pt").input_ids
#     example['labels'] = tokenizer(example["description"], padding="max_length",max_length=2048, truncation=True, return_tensors="pt").input_ids

#     return example

In [8]:
def tokenize_function(example):
    # Tokenize the "result" column as labels
    example['input_ids'] = tokenizer(example['code_snippet'], padding="max_length", truncation=True, return_tensors="pt")['input_ids']
    example['labels'] = tokenizer(example["description"], truncation=True,  max_length=4096, return_tensors="pt", padding='max_length')['input_ids']
    return example

In [9]:
tokenized_datasets = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/395 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [10]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['code_snippet', 'description', 'input_ids', 'labels'],
        num_rows: 395
    })
    test: Dataset({
        features: ['code_snippet', 'description', 'input_ids', 'labels'],
        num_rows: 99
    })
})

In [11]:
tokenized_datasets = tokenized_datasets.remove_columns(['code_snippet', 'description'])

In [12]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(395))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(99))

In [16]:
!pip install accelerate -U

In [18]:
!pip install transformers[torch]

In [20]:
!pip install accelerate>=0.20.1

In [19]:
from transformers import TrainingArguments

output_dir = 'checkpoints'

training_args = TrainingArguments(output_dir=output_dir,
                                  per_device_train_batch_size=8,
                                  per_device_eval_batch_size=8,
                                  learning_rate=5e-5,
                                  evaluation_strategy="epoch")

ImportError: Using the `Trainer` with `PyTorch` requires `accelerate>=0.20.1`: Please run `pip install transformers[torch]` or `pip install accelerate -U`

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

In [88]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [89]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)


In [91]:
outputs = model(input_ids=input_ids)
print(outputs.keys())

NameError: name 'inputs' is not defined

In [90]:
trainer.train()

ValueError: The model did not return a loss from the inputs, only the following keys: last_hidden_state. For reference, the inputs it received are input_ids.

In [49]:
print(model)

FunnelBaseModel(
  (embeddings): FunnelEmbeddings(
    (word_embeddings): Embedding(30522, 768)
    (layer_norm): LayerNorm((768,), eps=1e-09, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): FunnelEncoder(
    (attention_structure): FunnelAttentionStructure(
      (sin_dropout): Dropout(p=0.1, inplace=False)
      (cos_dropout): Dropout(p=0.1, inplace=False)
    )
    (blocks): ModuleList(
      (0-2): 3 x ModuleList(
        (0-3): 4 x FunnelLayer(
          (attention): FunnelRelMultiheadAttention(
            (hidden_dropout): Dropout(p=0.1, inplace=False)
            (attention_dropout): Dropout(p=0.1, inplace=False)
            (q_head): Linear(in_features=768, out_features=768, bias=False)
            (k_head): Linear(in_features=768, out_features=768, bias=True)
            (v_head): Linear(in_features=768, out_features=768, bias=True)
            (post_proj): Linear(in_features=768, out_features=768, bias=True)
            (layer_norm): La

In [5]:
!pip install accelerate -U

  Using cached torch-2.2.2-cp311-cp311-win_amd64.whl.metadata (26 kB)
  Using cached sympy-1.12-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/297.6 kB ? eta -:--:--
   ------------ --------------------------- 92.2/297.6 kB 5.1 MB/s eta 0:00:01
   ---------------------------- ----------- 215.0/297.6 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------  297.0/297.6 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------  297.0/297.6 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------  297.0/297.6 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------  297.0/297.6 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------  297.0/297.6 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------  297.0/297.6 kB 2.6 MB/s eta 0:00:01
   ---------------------------------------  297.0/297.6 kB 2.6 MB/s eta 0:00:01
   --------------


[notice] A new release of pip is available: 23.3.2 -> 24.0
[notice] To update, run: C:\Users\diya.sharma\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [8]:
# !pip install accelerate

In [14]:
!pip install accelerate>=0.21.0


In [22]:
import accelerate

print(f"Accelerate version: {accelerate.__version__}")

Accelerate version: 0.29.3


In [23]:
from transformers import TrainingArguments

output_dir = 'checkpoints'

training_args = TrainingArguments(output_dir=output_dir,
                                  per_device_train_batch_size=8,
                                  per_device_eval_batch_size=8,
                                  learning_rate=5e-5,
                                  evaluation_strategy="epoch")

In [24]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [7]:
!pip install evaluate

   ---------------------------------------- 0.0/84.1 kB ? eta -:--:--
   -------------------------------------- - 81.9/84.1 kB 1.5 MB/s eta 0:00:01
   -------------------------------------- - 81.9/84.1 kB 1.5 MB/s eta 0:00:01
   -------------------------------------- - 81.9/84.1 kB 1.5 MB/s eta 0:00:01
   ---------------------------------------- 84.1/84.1 kB 429.6 kB/s eta 0:00:00



[notice] A new release of pip is available: 23.3.2 -> 24.0
[notice] To update, run: C:\Users\diya.sharma\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [11]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

In [12]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)


NameError: name 'small_train_dataset' is not defined

In [12]:
trainer.train()

NameError: name 'trainer' is not defined

In [79]:
from datasets import load_dataset

dataset = load_dataset("Ankita802/code-description1")

Using the latest cached version of the dataset since Ankita802/code-description1 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\diya.sharma\.cache\huggingface\datasets\Ankita802___code-description1\default\0.0.0\f594dd591d5b375b931b10ea0d44e44786cca593 (last modified on Wed Apr 24 09:54:45 2024).


In [80]:
# from transformers import AutoTokenizer, FunnelForSequenceClassification
from transformers import FunnelForQuestionAnswering, FunnelTokenizer
tokenizer = FunnelTokenizer.from_pretrained("funnel-transformer/small-base")
model = FunnelForQuestionAnswering.from_pretrained("funnel-transformer/small-base")

Some weights of FunnelForQuestionAnswering were not initialized from the model checkpoint at funnel-transformer/small-base and are newly initialized: ['decoder.layers.0.attention.k_head.bias', 'decoder.layers.0.attention.k_head.weight', 'decoder.layers.0.attention.layer_norm.bias', 'decoder.layers.0.attention.layer_norm.weight', 'decoder.layers.0.attention.post_proj.bias', 'decoder.layers.0.attention.post_proj.weight', 'decoder.layers.0.attention.q_head.weight', 'decoder.layers.0.attention.r_kernel', 'decoder.layers.0.attention.r_r_bias', 'decoder.layers.0.attention.r_s_bias', 'decoder.layers.0.attention.r_w_bias', 'decoder.layers.0.attention.seg_embed', 'decoder.layers.0.attention.v_head.bias', 'decoder.layers.0.attention.v_head.weight', 'decoder.layers.0.ffn.layer_norm.bias', 'decoder.layers.0.ffn.layer_norm.weight', 'decoder.layers.0.ffn.linear_1.bias', 'decoder.layers.0.ffn.linear_1.weight', 'decoder.layers.0.ffn.linear_2.bias', 'decoder.layers.0.ffn.linear_2.weight', 'decoder.laye

In [81]:
def tokenize_function(batch):
    # Tokenize the "code_snippet" column and add tokenized inputs to the batch dictionary
    tokenized_code = tokenizer(batch['code_snippet'], truncation=True, padding='max_length', return_tensors="pt")
    batch['input_ids'] = tokenized_code['input_ids']
    batch['attention_mask'] = tokenized_code['attention_mask']
    batch['token_type_ids'] = tokenized_code['token_type_ids']
    
    
    # Tokenize the "description" column and add tokenized labels to the batch dictionary
    tokenized_description = tokenizer(batch["description"], truncation=True, padding='max_length', return_tensors="pt")
    batch['labels'] = tokenized_description['input_ids']  # Assuming 'labels' correspond to tokenized descriptions
    
    return batch


In [94]:
from transformers import FunnelForQuestionAnswering, FunnelTokenizer
import datasets  # Assuming you're using the datasets library for handling datasets

# Load your dataset
dataset = datasets.load_dataset("Ankita802/code-description1")

# Initialize Funnel tokenizer and model
tokenizer = FunnelTokenizer.from_pretrained("funnel-transformer/small-base")
model = FunnelForQuestionAnswering.from_pretrained("funnel-transformer/small-base")

# Assuming your dataset contains examples with 'code_snippet' and 'description' keys
# for example in dataset['train']:  # Adjust the split name as per your dataset structure
#     tokenized_input = tokenizer(example['code_snippet'], truncation=True, max_length=4096, return_tensors="pt", padding='max_length')
#     print("Tokenized Input IDs Shape:", tokenized_input['input_ids'].shape)
#     print("Tokenized Input IDs:", tokenized_input['input_ids'])
    
#     tokenized_labels = tokenizer(example['description'], truncation=True, max_length=4096, return_tensors="pt", padding='max_length')
#     print("Tokenized Labels Shape:", tokenized_labels['input_ids'].shape)
#     print("Tokenized Labels:", tokenized_labels['input_ids'])


Tokenized Input IDs Shape: torch.Size([1, 4096])
Tokenized Input IDs: tensor([[ 101, 2065, 2025,  ...,    0,    0,    0]])
Tokenized Labels Shape: torch.Size([1, 4096])
Tokenized Labels: tensor([[ 101, 2023, 3642,  ...,    0,    0,    0]])
Tokenized Input IDs Shape: torch.Size([1, 4096])
Tokenized Input IDs: tensor([[  101, 18750,  5950,  ...,     0,     0,     0]])
Tokenized Labels Shape: torch.Size([1, 4096])
Tokenized Labels: tensor([[ 101, 2023, 7526,  ...,    0,    0,    0]])
Tokenized Input IDs Shape: torch.Size([1, 4096])
Tokenized Input IDs: tensor([[ 101, 2026, 1035,  ...,    0,    0,    0]])
Tokenized Labels Shape: torch.Size([1, 4096])
Tokenized Labels: tensor([[ 101, 1996, 3642,  ...,    0,    0,    0]])
Tokenized Input IDs Shape: torch.Size([1, 4096])
Tokenized Input IDs: tensor([[ 101, 1001, 1996,  ...,    0,    0,    0]])
Tokenized Labels Shape: torch.Size([1, 4096])
Tokenized Labels: tensor([[ 101, 2023, 3642,  ...,    0,    0,    0]])
Tokenized Input IDs Shape: torch.S

In [92]:
# Assuming 'dataset' is your dataset containing examples
for example in dataset:
    tokenized_input = tokenizer(example['code_snippet'], truncation=True, max_length=4096, return_tensors="pt", padding='max_length')
    print("Tokenized Input IDs Shape:", tokenized_input['input_ids'].shape)
    print("Tokenized Input IDs:", tokenized_input['input_ids'])


TypeError: string indices must be integers, not 'str'

In [64]:
def tokenize_function(example):
    # Tokenize the "result" column as labels
    example['input_ids'] = tokenizer(example['code_snippet'], truncation=True, return_tensors="pt", padding='max_length')['input_ids']
    #example['labels'] = tokenizer(example["description"], truncation=True,  max_length=4096, return_tensors="pt", padding='max_length')['input_ids']
    # input_ids_length = len(example['input_ids'][0])
    # print(input_ids_length)
    # return {'input_ids': tokenized_inputs['input_ids'], 'attention_mask': tokenized_inputs['attention_mask']}


In [82]:
tokenized_datasets = dataset.map(tokenize_function)

Map:   0%|          | 0/395 [00:00<?, ? examples/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

In [83]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['code_snippet', 'description', 'input_ids', 'attention_mask', 'token_type_ids', 'labels'],
        num_rows: 395
    })
    test: Dataset({
        features: ['code_snippet', 'description', 'input_ids', 'attention_mask', 'token_type_ids', 'labels'],
        num_rows: 99
    })
})

In [84]:
tokenized_datasets = tokenized_datasets.remove_columns(['code_snippet', 'description', 'attention_mask', 'token_type_ids'])

In [85]:
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 395
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 99
    })
})

In [86]:
small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(395))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(99))

In [31]:
!pip install accelerate

In [15]:
import accelerate

print(f"Accelerate version: {accelerate.__version__}")

Accelerate version: 0.29.3


In [87]:
from transformers import TrainingArguments

output_dir = 'checkpoints_runs1'

training_args = TrainingArguments(output_dir=output_dir,
                                 per_device_train_batch_size=8,
                                 per_device_eval_batch_size=8,
                                 learning_rate=5e-5,
                                 evaluation_strategy="epoch")

In [88]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

In [89]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [90]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

In [91]:
trainer.train()

  0%|          | 0/150 [00:00<?, ?it/s]

RuntimeError: index 4 is out of bounds for dimension 0 with size 4

In [103]:
print(trainer.args.per_device_train_batch_size)

8


In [104]:
print(trainer.model.config)

FunnelConfig {
  "_name_or_path": "funnel-transformer/small-base",
  "activation_dropout": 0.0,
  "architectures": [
    "FunnelBaseModel"
  ],
  "attention_dropout": 0.1,
  "attention_type": "relative_shift",
  "block_repeats": [
    1,
    1,
    1
  ],
  "block_sizes": [
    4,
    4,
    4
  ],
  "d_head": 64,
  "d_inner": 3072,
  "d_model": 768,
  "hidden_act": "gelu_new",
  "hidden_dropout": 0.1,
  "initializer_range": 0.1,
  "initializer_std": null,
  "layer_norm_eps": 1e-09,
  "max_position_embeddings": 512,
  "model_type": "funnel",
  "n_head": 12,
  "num_decoder_layers": 2,
  "pool_q_only": true,
  "pooling_type": "mean",
  "problem_type": "single_label_classification",
  "rel_attn_type": "factorized",
  "separate_cls": true,
  "transformers_version": "4.40.0",
  "truncate_seq": true,
  "type_vocab_size": 3,
  "vocab_size": 30522
}

